# ADVC W1c: INT8 head padding check + smoke + calibrate (branch `revision-w1`, TinyImageNet, Kaggle T4)

After W1b (RESULTS_LOG R-002): the INT8 loader now pads layers whose out_features is not a multiple of 16 (the 200-way head -> 208) and slices the extra outputs off.

**Before running:** GPU T4 x2, Internet On, Add Input **jainmanav20/advc-results** and **albert1994/tinyimagenet**. Then **Save Version -> Save & Run All (Commit)**.

| Cell | What | Estimate (unmeasured) |
|---|---|---|
| 1-2 | clone, pins, link checkpoints, build tiny_if | ~3-5 min (CPU) |
| 3 | all unit tests, incl. new tests/test_int8_pad.py | ~1 min |
| 4 | P4: full DeiT-S INT8 padded vs unpadded at batch 16 (logits, input gradients; base and July AT ep7) + padded batch 32 | ~1-2 min GPU |
| 5 | smoke_tiny: fp32/int8/int4, one batch of 32, load/forward/backward/FGSM/PGD-20 | ~2-3 min GPU |
| 6 | calibrate_tiny: throughput at batch 16 and 32, fp32/int8/int4 (TF32 off: T4 has none) | ~2-4 min GPU |
| 7 | summary (not truncated): copy it back to Claude | - |

Total: ~6-9 min GPU (approved 2026-11-11), ~12-18 min wall. Times are estimates.

In [ ]:
# 1. Setup: clone branch, pins, link checkpoints, helper
import os, subprocess, sys, time, glob, json
BRANCH = 'revision-w1'
W = '/kaggle/working/w1c'
OUT_ROOT = W + '/results'
os.makedirs(W, exist_ok=True)
if not os.path.isdir('/kaggle/working/ADVC'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Jmanav/ADVC.git', '/kaggle/working/ADVC'], check=True)
os.chdir('/kaggle/working/ADVC')
GIT = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()
print('git:', GIT)
!pip install -q --no-deps bitsandbytes==0.50.2
!pip install -q torchattacks pytest
import torch, bitsandbytes
print('torch', torch.__version__, '| bitsandbytes', bitsandbytes.__version__, '|', torch.cuda.get_device_name(0))
assert bitsandbytes.__version__ == '0.50.2'
hits = glob.glob('/kaggle/input/**/at_int8_epoch01.pt', recursive=True)
assert hits, 'No checkpoints found: attach jainmanav20/advc-results'
CKPT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs('results/checkpoints', exist_ok=True)
if not os.path.exists('results/checkpoints/tiny-imagenet'):
    os.symlink(CKPT_ROOT, 'results/checkpoints/tiny-imagenet')
assert os.path.isfile('results/checkpoints/tiny-imagenet/base/deit_small_ft.pt')

TIMES = {}
def sh(name, cmd, env=None):
    t0 = time.time()
    with open(f'{W}/{name}.log', 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, **(env or {})))
        for line in p.stdout:
            log.write(line); print(line, end='')
        p.wait()
    TIMES[name] = {'seconds': round(time.time() - t0, 1), 'exit': p.returncode}
    print('[' + name + ']', TIMES[name])
    return p.returncode

In [ ]:
# 2. Build /kaggle/working/tiny_if from albert1994/tinyimagenet
SRC = '/kaggle/input/datasets/albert1994/tinyimagenet/tiny-imagenet-200'
if not os.path.isdir(SRC):
    cands = [p for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True)
             if os.path.isfile(os.path.join(p, 'val', 'val_annotations.txt'))]
    assert cands, 'TinyImageNet not found: attach albert1994/tinyimagenet'
    SRC = cands[0]
if not os.path.isdir('/kaggle/working/tiny_if/val_if'):
    sh('prepare_tiny', [sys.executable, 'scripts/prepare_tiny_imagenet.py', '--root', SRC, '--out', '/kaggle/working/tiny_if'])
assert os.path.isdir('/kaggle/working/tiny_if/val_if'), 'val_if was not built'
print('val classes:', len(os.listdir('/kaggle/working/tiny_if/val_if')))

In [ ]:
# 3. All unit tests (CPU + GPU)
sh('pytest_all', [sys.executable, '-m', 'pytest', '-q', '-rs', 'tests/'])

In [ ]:
%%writefile /kaggle/working/w1c/p4_full_model.py
# P4: full DeiT-S INT8, padded head (default) vs unpadded (int8_pad_multiple=0), on 16 real val images
# (batch 16 works for both). Base model and July AT INT8 epoch-7 checkpoint. Then padded at batch 32.
import json, sys, traceback
sys.path.insert(0, '.')
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from models.loader import load_config, load_model
from utils.datasets import build_eval_transform, build_remapped_folder

cfg = load_config('configs/base.yaml')
ds = build_remapped_folder(cfg['dataset']['val_dir'], build_eval_transform(cfg), cfg)
idx32 = list(range(0, len(ds), len(ds) // 32))[:32]
X, Y = next(iter(DataLoader(Subset(ds, idx32), batch_size=32, shuffle=False)))
X, Y = X.cuda(), Y.cuda()
x16, y16 = X[:16], Y[:16]

def logits(m, x):
    o = m(x)
    return o.logits if hasattr(o, 'logits') else o

def grad(m, x, y):
    xg = x.clone().requires_grad_(True)
    loss = F.cross_entropy(logits(m, xg).float(), y)
    g, = torch.autograd.grad(loss, xg)
    return g

out = {}
for tag, ck in [('base', None), ('at_int8_ep7', 'results/checkpoints/tiny-imagenet/at/at_int8_epoch07.pt')]:
    r = {}
    try:
        sd_p = torch.load(ck, map_location='cuda') if ck else None
        sd_u = torch.load(ck, map_location='cuda') if ck else None
        mp = load_model('deit_small', 'int8', cfg, device='cuda', int8_state_dict=sd_p).eval()
        mu = load_model('deit_small', 'int8', cfg, device='cuda', int8_state_dict=sd_u, int8_pad_multiple=0).eval()
        r['head_out_features_padded_vs_unpadded'] = [mp.head.out_features, mu.head.out_features]
        with torch.no_grad():
            lp, lu = logits(mp, x16), logits(mu, x16)
        r['logits_shape'] = list(lp.shape)
        r['logits_bit_equal'] = bool(torch.equal(lp, lu))
        r['logits_max_abs_diff'] = float((lp.float() - lu.float()).abs().max())
        r['pred_agree'] = float((lp.argmax(1) == lu.argmax(1)).float().mean())
        r['acc16_padded_unpadded'] = [float((lp.argmax(1) == y16).float().mean()), float((lu.argmax(1) == y16).float().mean())]
        gp, gu = grad(mp, x16, y16), grad(mu, x16, y16)
        r['grad_bit_equal'] = bool(torch.equal(gp, gu))
        r['grad_max_abs_diff'] = float((gp - gu).abs().max())
        r['grad_rel_max_diff'] = float((gp - gu).abs().max() / gu.abs().max().clamp_min(1e-30))
        r['grad_sign_agree'] = float((torch.sign(gp) == torch.sign(gu)).float().mean())
        try:
            with torch.no_grad():
                l32 = logits(mp, X)
            g32 = grad(mp, X, Y)
            r['padded_bs32'] = {'ok': True, 'acc32': float((l32.argmax(1) == Y).float().mean()), 'grad_finite': bool(torch.isfinite(g32).all())}
        except Exception as e:
            r['padded_bs32'] = {'ok': False, 'error': repr(e)[:300]}
        del mp, mu
        torch.cuda.empty_cache()
    except Exception:
        r['error'] = traceback.format_exc()[-3000:]
    out[tag] = r
    print(tag, json.dumps(r))
json.dump(out, open('/kaggle/working/w1c/p4_full_model.json', 'w'), indent=1)
print('P4JSON ' + json.dumps(out))

In [ ]:
# 4. Run P4 (~1-2 min GPU)
sh('p4_full_model', [sys.executable, f'{W}/p4_full_model.py'])

In [ ]:
# 5. Smoke test, all precisions (~2-3 min GPU)
sh('smoke_tiny', [sys.executable, 'run.py', '--exp', 'configs/exp/smoke_tiny.yaml', '--out-root', OUT_ROOT])

In [ ]:
# 6. Calibrate (~2-4 min GPU)
sh('calibrate_tiny', [sys.executable, 'run.py', '--exp', 'configs/exp/calibrate_tiny.yaml', '--out-root', OUT_ROOT])

In [ ]:
# 7. SUMMARY: copy everything this cell prints back to Claude (also /kaggle/working/w1c/summary.txt). Not truncated.
L = ['git: ' + GIT, 'torch ' + torch.__version__ + ' | bnb ' + bitsandbytes.__version__ + ' | ' + torch.cuda.get_device_name(0)]
L.append('== wall times / exit codes ==')
L.append(json.dumps(TIMES, indent=1))
L.append('== pytest_all (last 30 lines) ==')
L += [l.rstrip() for l in open(f'{W}/pytest_all.log').readlines()[-30:]]
L.append('== P4 full model ==')
p = f'{W}/p4_full_model.json'
L.append(open(p).read() if os.path.exists(p) else 'no JSON; log tail:\n' + ''.join(open(f'{W}/p4_full_model.log').readlines()[-40:]))
for f in sorted(glob.glob(f'{OUT_ROOT}/tiny-imagenet/runs/*.json')):
    d = json.load(open(f))
    L.append('== run ' + os.path.basename(f) + ' exp=' + d['exp']['name'] + ' status=' + str(d.get('status')) + ' ==')
    for k in ['timings', 'results', 'sections', 'warnings']:
        L.append('-- ' + k + ' --')
        L.append(json.dumps(d.get(k), indent=1))
    L.append('-- error steps --')
    L.append(json.dumps([e.get('step') for e in d.get('errors', [])]))
for f in sorted(glob.glob(f'{OUT_ROOT}/tiny-imagenet/runs/*.traceback.txt')):
    L.append('== traceback ' + os.path.basename(f) + ' ==')
    L.append(open(f).read()[-6000:])
for name in ['smoke_tiny', 'calibrate_tiny']:
    L.append('== ' + name + ' console (last 40 lines) ==')
    L += [l.rstrip() for l in open(f'{W}/{name}.log').readlines()[-40:]]
text = '\n'.join(L)
open(f'{W}/summary.txt', 'w').write(text)
print(text)